# Multilingual RAG retrieval

Cross-lingual retrieval using **multilingual-e5** embeddings and **LangChain + Chroma**.

Corpus: UDHR preamble + articles in English and Hindi (`data/udhr_en.txt`, `data/udhr_hi.txt`).

**Prerequisites** (from repo root):

```bash
uv sync
uv run jupyter lab experiments/multilingual-rag-retrieval/multilingual_e5_chroma.ipynb
```

Run cells top to bottom.

In [34]:
from pathlib import Path

# Folder paths setup
EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "multilingual_e5_chroma.ipynb").exists():
    alt = Path("experiments/multilingual-rag-retrieval").resolve()
    if (alt / "multilingual_e5_chroma.ipynb").exists():
        EXPERIMENT_DIR = alt

DATA_DIR = EXPERIMENT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Setup + ingest

In [35]:
import re
import shutil
from pathlib import Path
from typing import Any

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


CHROMA_PATH = EXPERIMENT_DIR / "chroma"
COLLECTION_NAME = "multilingual_rag_facts"
MODEL_NAME = "intfloat/multilingual-e5-base"

LANG_SOURCE_MAP = {
    "en": "udhr_en.txt",
    "hi": "udhr_hi.txt",
}

HI_ARTICLE_NUM = r"[०-९\d]+"
ARTICLE_MARKERS = {
    "en": re.compile(r"(?=\bArticle\s+(\d+)\b)", re.IGNORECASE),
    "hi": re.compile(rf"(?=अनुच्छेद\s*({HI_ARTICLE_NUM})\.?)"),
}
ARTICLE_ID_FROM_START = {
    "en": re.compile(r"^\s*Article\s+(\d+)\b", re.IGNORECASE),
    "hi": re.compile(rf"^\s*अनुच्छेद\s*({HI_ARTICLE_NUM})\.?"),
}
DEVANAGARI_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")


class E5Embeddings(Embeddings):
    """multilingual-e5 requires query:/passage: prefixes."""

    def __init__(self, model_name: str = MODEL_NAME) -> None:
        self._model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        prefixed = [f"passage: {t}" for t in texts]
        vectors = self._model.encode(
            prefixed, normalize_embeddings=True, show_progress_bar=True
        )
        return vectors.tolist()

    def embed_query(self, text: str) -> list[float]:
        vectors = self._model.encode(
            [f"query: {text}"], normalize_embeddings=True, show_progress_bar=False
        )
        return vectors[0].tolist()


def _normalize_article_num(raw: str) -> str:
    return raw.translate(DEVANAGARI_DIGITS)


def _article_id(lang: str, chunk: str) -> str:
    pattern = ARTICLE_ID_FROM_START.get(lang)
    if pattern:
        match = pattern.search(chunk.strip())
        if match:
            return _normalize_article_num(match.group(1))
    return "preamble"


def load_lang_text(lang: str) -> str:
    path = DATA_DIR / LANG_SOURCE_MAP[lang]
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    return path.read_text(encoding="utf-8")


def split_by_articles(lang: str, text: str) -> list[Document]:
    marker = ARTICLE_MARKERS.get(lang)
    if marker:
        parts = marker.split(text)
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) > 1:
            docs = []
            for part in parts:
                article = _article_id(lang, part)
                docs.append(
                    Document(
                        page_content=part,
                        metadata={"lang": lang, "article": article, "source": "udhr"},
                    )
                )
            return docs

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    return splitter.create_documents(
        [text],
        metadatas=[{"lang": lang, "article": "unknown", "source": "udhr"}],
    )


def build_documents(langs: list[str]) -> list[Document]:
    all_docs: list[Document] = []
    for lang in langs:
        text = load_lang_text(lang)
        if not text.strip():
            print(f"{lang}: 0 chars extracted — check source file")
        chunks = split_by_articles(lang, text)
        print(f"{lang}: {len(chunks)} chunks")
        all_docs.extend(chunks)
    return all_docs


def build_vectorstore(documents: list[Document], *, reset: bool = True) -> Chroma:
    if reset and CHROMA_PATH.exists():
        shutil.rmtree(CHROMA_PATH)
    CHROMA_PATH.mkdir(parents=True, exist_ok=True)
    embeddings = E5Embeddings()
    return Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(CHROMA_PATH),
    )


print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Chroma path: {CHROMA_PATH}")

Experiment dir: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/multilingual-rag-retrieval
Chroma path: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/multilingual-rag-retrieval/chroma


In [36]:
# Languages to ingest: English + Hindi
langs = ["en", "hi"]

documents = build_documents(langs)
vectorstore = build_vectorstore(documents, reset=True)
print(f"Indexed {len(documents)} chunks into '{COLLECTION_NAME}'")

en: 61 chunks
hi: 61 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Indexed 122 chunks into 'multilingual_rag_facts'


## 2. Search

Change `query` and `query_lang`, then run the cell. A hit in another language with the same article is cross-lingual retrieval.

In [21]:
query = "Everyone has the right to life, liberty and security of person."
query_lang = "en"  # en | hi — language of the query text
k = 5

results = vectorstore.similarity_search_with_score(query, k=k)

rows = []
for rank, (doc, score) in enumerate(results, start=1):
    doc_lang = doc.metadata.get("lang")
    rows.append(
        {
            "rank": rank,
            "article": doc.metadata.get("article"),
            "lang": doc_lang,
            "cross_lingual": doc_lang != query_lang,
            "score": round(float(score), 4),
            "text": doc.page_content[:120] + "...",
        }
    )

display(pd.DataFrame(rows))

,rank,article,lang,score,text
0,1,3,en,0.2450,"Article 3 \nEveryone has the right to life, l..."
1,2,22,en,0.2827,"Article 22 \nEveryone, as a member of society..."
2,3,2,en,0.2835,Article 2 \nEveryone is entitled to all the r...
3,4,25,en,0.3057,Article 25 \n1. Everyone has the right to a s...
4,5,13,en,0.3084,Article 13 \n1. Everyone has the right to fre...
